# 03 — Silver (typed, conformed)

Shreds every bronze VARIANT into a typed `silver_<entity>` table for all entities, driven by the `(path, alias, type)` maps in `synergy_schemas.SILVER_COLUMNS` (verified against the OpenAPI spec).

> Loads incrementally with `MERGE` (upsert on the natural key): a re-run updates changed rows and inserts new ones rather than rewriting the table, and `MERGE WITH SCHEMA EVOLUTION` absorbs new columns if you extend `SILVER_COLUMNS`. This runs over Databricks Connect and in the workspace alike.

**The conformance key:** `teams.external_id_mlbam` (and `games.external_id_mlbam`) is the MLBAM id, the standard cross-source join key, so this Synergy data lines up with any other MLBAM-keyed data the customer has. It's also the spine of the gold star schema (notebook 05).

In [ ]:
# Job parameter bridge: when run as a Databricks Job task, the bundle's base_parameters arrive as
# notebook widgets. Mirror them into os.environ before the setup cell reads them, so the same
# os.getenv(...) logic works as a job, in a workspace, or locally via .env. dbutils is absent under
# local Databricks Connect, so the except clause handles that case.
import os
for _k in ("UC_CATALOG", "UC_SCHEMA"):
    try:
        _v = dbutils.widgets.get(_k)            # noqa: F821 — dbutils is injected in Databricks
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass

In [ ]:
# Dual-mode setup: runs locally via Databricks Connect and inside a Databricks Git Folder.
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
from synergy_schemas import SILVER_COLUMNS, PRIMARY_KEYS, ENTITIES

def upsert(table: str, select_sql: str, keys: list):
    """Create `table` from the projection if absent, then MERGE the (already deduped) select into it on
    `keys`. Re-runnable: matched keys are updated in place, new keys inserted. WITH SCHEMA EVOLUTION lets
    a widened SILVER_COLUMNS add columns without a rebuild. (autoMerge config is unavailable on serverless,
    so schema evolution is requested per statement.)"""
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} AS SELECT * FROM ({select_sql}) WHERE 1=0")
    on = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
    spark.sql(f"""
        MERGE WITH SCHEMA EVOLUTION INTO {table} t
        USING ({select_sql}) s
        ON {on}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def build_silver(entity: str):
    cols  = SILVER_COLUMNS[entity]
    table = f"{S}.silver_{entity}"
    # Some entities land no data on a run: empty for this org (e.g. practice_training_workout) or a
    # transient source 5xx leaves bronze absent. Ensure an empty, correctly-typed shell exists so
    # downstream (04, 05, 06) resolves the table. CREATE IF NOT EXISTS never wipes a populated silver
    # if bronze is temporarily missing.
    if not spark.catalog.tableExists(f"{B}.bronze_{entity}"):
        empty = ",\n            ".join(f"cast(null AS {typ}) AS {alias}" for _, alias, typ in cols)
        spark.sql(f"CREATE TABLE IF NOT EXISTS {table} AS "
                  f"SELECT {empty}, cast(null AS timestamp) AS _ingestion_timestamp LIMIT 0")
        print(f"  silver_{entity:26} {'0':>7}  (empty shell — no bronze data)")
        return
    # VARIANT navigation: `data:a:b::type` (nested objects use `:`, e.g. data:homeTeam:division:id).
    # Clean values cast directly; if an entity has junk that hard-fails a cast, swap `data:{path}::{typ}`
    # for try_cast(data:{path}::string AS {typ}) to null bad values instead.
    projection = ",\n            ".join(f"data:{path}::{typ} AS {alias}" for path, alias, typ in cols)
    pk = PRIMARY_KEYS[entity]
    # Dedup to one row per natural key (latest ingest wins) so MERGE sees a single match. QUALIFY filters
    # on the window result without a wrapping subquery, and can reference the projection aliases directly.
    select_sql = f"""
        SELECT {projection}, _ingestion_timestamp
        FROM {B}.bronze_{entity}
        QUALIFY row_number() OVER (PARTITION BY {','.join(pk)} ORDER BY _ingestion_timestamp DESC) = 1
    """
    upsert(table, select_sql, pk)
    print(f"  silver_{entity:26} {spark.table(table).count():>7,} rows  ({len(cols)} cols)")

for e in ENTITIES:
    try:
        build_silver(e["name"])
    except Exception as ex:
        print(f"  silver_{e['name']:26} SKIP/err: {str(ex)[:70]}")

In [ ]:
# sample + the cross-source conformance key
spark.sql(f"SELECT id, name, external_id_mlbam, league_name, division_name FROM {S}.silver_teams LIMIT 5").show(truncate=False)
spark.sql(f"SELECT id, season, game_date, home_team_name, away_team_name, game_level FROM {S}.silver_games LIMIT 5").show(truncate=False)